# 03 — PennyLane Lightning: differentiable Ising/VQE introduction

**Level:** intermediate. **Environment:** PennyLane 0.45-compatible CPU `lightning.qubit`. **GPU extension:** `lightning.gpu` via separately validated Slurm/Apptainer image, not this Kubernetes notebook. **Run time:** seconds locally. **Outputs:** energy and analytical gradient.

An SDK (PennyLane), a simulator backend (Lightning), a device allocation (H200) and a physical quantum processor are four distinct concepts.

In [ ]:
import pennylane as qml
from pennylane import numpy as pnp

dev = qml.device("lightning.qubit", wires=2)
hamiltonian = qml.PauliZ(0) + 0.5 * (qml.PauliX(0) @ qml.PauliX(1))

@qml.qnode(dev, interface="autograd", diff_method="best")
def energy(theta):
    qml.RY(theta, wires=0)
    qml.CNOT(wires=[0, 1])
    return qml.expval(hamiltonian)

theta = pnp.array(0.37, requires_grad=True)
E = energy(theta)
grad = qml.grad(energy)(theta)
print("Ising energy:", float(E), "gradient:", float(grad))
assert pnp.isfinite(E) and pnp.isfinite(grad)

## Extension

A real GPU run requires an approved resource offering, a separate validated `pennylane-lightning-gpu` image, native `sbatch`, `srun apptainer exec --nv` and cross-check against the same CPU Hamiltonian. No implicit GPU fallback. See the M5 image catalogue and first-gate acceptance plan.